In [3]:
!pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 5.0 MB/s  0:00:07m0:00:0100:01


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2
from Bio import SeqIO

Подгрузим посл-ть из первой хромосомы человека:

In [6]:
record = SeqIO.read("chr1.fasta", "fasta")
real_seq = str(record.seq[:100000]).upper()

Также создадим случайную последовательность с теми же частотами нуклеотидов:

In [9]:
nucleotides = ['A', 'C', 'G', 'T']
freqs = [
    real_seq.count('A') / len(real_seq),
    real_seq.count('C') / len(real_seq),
    real_seq.count('G') / len(real_seq),
    real_seq.count('T') / len(real_seq)
]

freqs = np.array(freqs)
freqs = freqs / freqs.sum() #нормировка частот

np.random.seed(42)
random_seq = ''.join(np.random.choice(nucleotides, size=len(real_seq), p=freqs))

Напишем функцию для подсчета наблюдаемых частот динуклеотидов:

In [10]:
def observed_dinuc_counts(seq):
    counts = np.zeros((4, 4), dtype=int)
    for i in range(len(seq)-1):
        if seq[i] in nucleotides and seq[i+1] in nucleotides:
            from_idx = nucleotides.index(seq[i])
            to_idx = nucleotides.index(seq[i+1])
            counts[from_idx, to_idx] += 1
    return counts

А также функцию для расчета ожидаемых частот:

In [11]:
def expected_dinuc_counts(seq):
    N = len(seq) - 1 #кол-во динуклеотидов
    p_single = np.array([seq.count(nuc) for nuc in nucleotides]) / len(seq)
    expected = np.outer(p_single, p_single) * N
    return expected

И ещё функцию для статистики $\chi^2$:

In [12]:
def chi2_test(seq):
    observed = observed_dinuc_counts(seq)
    expected = expected_dinuc_counts(seq)
    
    chi2_stat = np.sum((observed - expected)**2 / expected)
    
    p_value = 1 - chi2.cdf(chi2_stat, df=9)
    
    return chi2_stat, p_value, observed, expected

**Исследуем реальную последовательность:**

In [13]:
chi2_real, p_real, obs_real, exp_real = chi2_test(real_seq)

print("Наблюдаемые частоты динуклеотидов (реальная посл-ть):")
print("     A    C    G    T")
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  {obs_real[i,0]:5d} {obs_real[i,1]:5d} {obs_real[i,2]:5d} {obs_real[i,3]:5d}")

print("Ожидаемые частоты:")
print("     A      C      G      T")
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  {exp_real[i,0]:6.1f} {exp_real[i,1]:6.1f} {exp_real[i,2]:6.1f} {exp_real[i,3]:6.1f}")

print(f"Хи-квадрат: {chi2_real:.2f}")
print(f"p-value: {p_real:.2e}")

Наблюдаемые частоты динуклеотидов (реальная посл-ть):
     A    C    G    T
A   8989  4786  6253  6707
C   6879  5461  1101  6482
G   5208  4143  4749  4183
T   5659  5534  6180  7685
Ожидаемые частоты:
     A      C      G      T
A  7147.5 5326.6 4887.9 6699.2
C  5326.6 3969.6 3642.7 4992.5
G  4887.9 3642.7 3342.6 4581.3
T  6699.2 4992.5 4581.3 6279.0
Хи-квадрат: 5950.09
p-value: 0.00e+00


*Вывод:*

Для реальной последовательности p-value < 0.5. Это значит, что гипотеза о независимости отвергается. Т.е. существует зависимость между соседними нуклеотидами, что логично для реальной последовательности хромосомы человека. Собственно, нужна Марковская модель 1-го порядка.

In [17]:
print("\n" + "="*50)
print("ИТОГОВЫЙ ВЫВОД")
print("="*50)
print(f"Реальная последовательность: p-value = {p_real:.2e} {'(< 0.05)' if p_real < 0.05 else '(≥ 0.05)'}")
print(f"Случайная последовательность: p-value = {p_rand:.4f} {'(< 0.05)' if p_rand < 0.05 else '(≥ 0.05)'}")
print("\nСтатистический вывод:")
if p_real < 0.05:
    print("- Для реальной последовательности гипотеза независимости ОТВЕРГАЕТСЯ")
    print("  → Существует зависимость между соседними нуклеотидами")
    print("  → Нужна марковская модель 1-го порядка")
else:
    print("- Для реальной последовательности гипотеза независимости НЕ ОТВЕРГАЕТСЯ")
    print("  → Модели 0-го порядка достаточно")

if p_rand >= 0.05:
    print("- Для случайной последовательности гипотеза независимости НЕ ОТВЕРГАЕТСЯ")
    print("  → Генератор работает правильно")
else:
    print("- Для случайной последовательности гипотеза независимости ОТВЕРГАЕТСЯ (странно!)")
    print("  → Возможно, недостаточно данных или проблемы с генератором")


ИТОГОВЫЙ ВЫВОД
Реальная последовательность: p-value = 0.00e+00 (< 0.05)
Случайная последовательность: p-value = 0.0856 (≥ 0.05)

Статистический вывод:
- Для реальной последовательности гипотеза независимости ОТВЕРГАЕТСЯ
  → Существует зависимость между соседними нуклеотидами
  → Нужна марковская модель 1-го порядка
- Для случайной последовательности гипотеза независимости НЕ ОТВЕРГАЕТСЯ
  → Генератор работает правильно


**Исследуем случайную последовательность:**

In [14]:
chi2_rand, p_rand, obs_rand, exp_rand = chi2_test(random_seq)

print("Наблюдаемые частоты динуклеотидов (случайная последовательность):")
print("     A    C    G    T")
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  {obs_rand[i,0]:5d} {obs_rand[i,1]:5d} {obs_rand[i,2]:5d} {obs_rand[i,3]:5d}")

print("\nОжидаемые частоты:")
print("     A      C      G      T")
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  {exp_rand[i,0]:6.1f} {exp_rand[i,1]:6.1f} {exp_rand[i,2]:6.1f} {exp_rand[i,3]:6.1f}")

print(f"Хи-квадрат статистика: {chi2_rand:.2f}")
print(f"p-value: {p_rand:.4f}")

Наблюдаемые частоты динуклеотидов (случайная последовательность):
     A    C    G    T
A   8905  6573  6119  8178
C   6406  4946  4547  6162
G   6069  4420  4224  5710
T   8395  6122  5533  7690

Ожидаемые частоты:
     A      C      G      T
A  8865.4 6568.9 6080.9 8259.5
C  6568.9 4867.3 4505.7 6119.9
G  6080.9 4505.7 4170.9 5665.3
T  8259.5 6119.9 5665.3 7695.0
Хи-квадрат статистика: 15.20
p-value: 0.0856


*Вывод:*

Для случайной последовательности p-value > 0.5. Это значит, что гипотеза о независимости не отвергается. Т.е. наш случайный генератор работает корректно.